In [1]:
# 05 - Modelado LSTM + sentimiento (forecast 24h)
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
import json

# Auto-detección de entorno: Colab o local
IS_COLAB = False
try:
    from google.colab import drive  # type: ignore
    IS_COLAB = True
except Exception:
    IS_COLAB = False

if IS_COLAB:
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2')
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name.lower() == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent

GOLD = PROJECT_ROOT / 'data/gold'
MODELS = PROJECT_ROOT / 'models'
REPORTS = PROJECT_ROOT / 'reports' / 'backtest'
MODELS.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

print('IS_COLAB:', IS_COLAB)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('GOLD:', GOLD)
print('MODELS:', MODELS)

Mounted at /content/drive
IS_COLAB: True
PROJECT_ROOT: /content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2
GOLD: /content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2/data/gold
MODELS: /content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2/models


In [2]:
# Importar librerías de modelado
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print('TensorFlow version:', tf.__version__)

TensorFlow version: 2.19.0


In [3]:
# Cargar Gold y preparar dataset
gold_path = GOLD / 'gold_btc_features_1h.csv'
if not gold_path.exists():
    raise FileNotFoundError(f'No existe {gold_path}. Ejecuta primero el notebook 04.')

df = pd.read_csv(gold_path)
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
df = df.sort_values('timestamp').reset_index(drop=True)

target_col = 'target_ret_24h'
drop_cols = [
    'timestamp', 'source', 'symbol', 'interval',
    'target_close_t_plus_24h', 'target_ret_24h', 'target_direction_24h',
    'investment_signal_24h',
]

feature_cols = [c for c in df.columns if c not in drop_cols and pd.api.types.is_numeric_dtype(df[c])]
df_model = df[['timestamp', target_col] + feature_cols].copy()
df_model = df_model.replace([np.inf, -np.inf], np.nan)
df_model[feature_cols] = df_model[feature_cols].ffill().bfill()
df_model = df_model.dropna(subset=[target_col])

print('Filas modelado:', len(df_model))
print('N features:', len(feature_cols))

Filas modelado: 976
N features: 30


In [4]:
# Construcción de secuencias para LSTM
LOOKBACK = 72  # 72 horas
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

X_all = df_model[feature_cols].values
y_all = df_model[target_col].values
t_all = df_model['timestamp'].values

def create_sequences(X, y, t, lookback):
    Xs, ys, ts = [], [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i-lookback:i])
        ys.append(y[i])
        ts.append(t[i])
    return np.array(Xs), np.array(ys), np.array(ts)

X_seq, y_seq, t_seq = create_sequences(X_all, y_all, t_all, LOOKBACK)
n = len(X_seq)
train_end = int(n * TRAIN_RATIO)
val_end = int(n * (TRAIN_RATIO + VAL_RATIO))

X_train, y_train = X_seq[:train_end], y_seq[:train_end]
X_val, y_val = X_seq[train_end:val_end], y_seq[train_end:val_end]
X_test, y_test = X_seq[val_end:], y_seq[val_end:]
t_test = t_seq[val_end:]

# Escalado (fit solo en train)
x_scaler = StandardScaler()
X_train_2d = X_train.reshape(-1, X_train.shape[-1])
x_scaler.fit(X_train_2d)

def transform_3d(X, scaler):
    X2d = X.reshape(-1, X.shape[-1])
    Xs = scaler.transform(X2d)
    return Xs.reshape(X.shape)

X_train_s = transform_3d(X_train, x_scaler)
X_val_s = transform_3d(X_val, x_scaler)
X_test_s = transform_3d(X_test, x_scaler)

y_scaler = StandardScaler()
y_train_s = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
y_val_s = y_scaler.transform(y_val.reshape(-1, 1)).ravel()
y_test_s = y_scaler.transform(y_test.reshape(-1, 1)).ravel()

print('Train:', X_train_s.shape, 'Val:', X_val_s.shape, 'Test:', X_test_s.shape)

Train: (632, 72, 30) Val: (136, 72, 30) Test: (136, 72, 30)


In [5]:
# Definir y entrenar modelo LSTM
tf.keras.utils.set_random_seed(42)

model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_train_s.shape[1], X_train_s.shape[2])),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1),
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
)

history = model.fit(
    X_train_s, y_train_s,
    validation_data=(X_val_s, y_val_s),
    epochs=60,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1,
    shuffle=False,
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 72, 64)         │        24,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 72, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 37,281 (145.63 KB)

 Trainable params: 37,281 (145.63 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 63ms/step - loss: 1.2812 - mae: 0.8339 - val_loss: 1.2276 - val_mae: 0.8751
Epoch 2/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 1.0397 - mae: 0.7357 - val_loss: 1.1516 - val_mae: 0.8398
Epoch 3/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.8614 - mae: 0.6724 - val_loss: 1.1223 - val_mae: 0.8182
Epoch 4/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.6958 - mae: 0.5967 - val_loss: 1.1688 - val_mae: 0.8143
Epoch 5/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.5506 - mae: 0.5259 - val_loss: 1.2340 - val_mae: 0.8388
Epoch 6/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.4653 - mae: 0.4701 - val_loss: 1.3482 - val_mae: 0.8247
Epoch 7/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.4533 - mae: 0.4709 - val_loss: 1.3282 - val_mae: 0.8832
Epoch 8/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.4103 - mae: 0.4559 - val_loss: 1.2593 - val_mae: 0.8338
Epoch 9/60
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.331

In [6]:
# Evaluación, guardado de artefactos y predicciones
y_pred_s = model.predict(X_test_s, verbose=0).ravel()
y_pred = y_scaler.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
direction_acc = ((y_pred > 0) == (y_test > 0)).mean()

print(f'MAE test: {mae:.6f}')
print(f'RMSE test: {rmse:.6f}')
print(f'Directional accuracy: {direction_acc:.4f}')

model_path = MODELS / 'lstm_btc_sentiment_24h.keras'
model.save(model_path)

pred_df = pd.DataFrame({
    'timestamp': pd.to_datetime(t_test, utc=True),
    'y_true_ret_24h': y_test,
    'y_pred_ret_24h': y_pred,
})
pred_df['y_true_direction'] = (pred_df['y_true_ret_24h'] > 0).astype(int)
pred_df['y_pred_direction'] = (pred_df['y_pred_ret_24h'] > 0).astype(int)
pred_df.to_csv(REPORTS / 'lstm_test_predictions_24h.csv', index=False)

meta = {
    'model_name': 'lstm_btc_sentiment_24h',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'lookback': LOOKBACK,
    'n_features': len(feature_cols),
    'features': feature_cols,
    'train_samples': int(len(X_train_s)),
    'val_samples': int(len(X_val_s)),
    'test_samples': int(len(X_test_s)),
    'metrics': {
        'mae': float(mae),
        'rmse': float(rmse),
        'directional_accuracy': float(direction_acc),
    },
    'files': {
        'model': str(model_path),
        'predictions': str(REPORTS / 'lstm_test_predictions_24h.csv'),
    },
}

with open(MODELS / 'lstm_btc_sentiment_24h_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print('Modelo guardado en:', model_path)
print('Predicciones guardadas en:', REPORTS / 'lstm_test_predictions_24h.csv')
pred_df.tail(5)

MAE test: 0.017296
RMSE test: 0.019834
Directional accuracy: 0.7941
Modelo guardado en: /content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2/models/lstm_btc_sentiment_24h.keras
Predicciones guardadas en: /content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2/reports/backtest/lstm_test_predictions_24h.csv


,timestamp,y_true_ret_24h,y_pred_ret_24h,y_true_direction,y_pred_direction
131,2026-03-10 01:00:00+00:00,0.010889,0.009825,1,1
132,2026-03-10 02:00:00+00:00,-0.006511,0.010133,0,1
133,2026-03-10 03:00:00+00:00,0.000208,0.009645,1,1
134,2026-03-10 04:00:00+00:00,0.001742,0.009446,1,1
135,2026-03-10 05:00:00+00:00,0.002245,0.008623,1,1
